# CT4101 Machine Learning 

## Machine Learning Portfolio: Predicting 30-Day Hospital Readmission in Patients with Diabetes Using Machine Learning

## 4. Preprocessing

### Student Name: Faith Greene

### Student ID: 26256593


This notebook performs the data cleaning & preparations, based on the findings of the audit.

#### Data Types

Variable types should be converted where necessary.

In [18]:
raw_data[string_variables] = raw_data[string_variables].astype("string")
raw_data[numerical_variables] = raw_data[numerical_variables].apply(pd.to_numeric, errors = "coerce")
raw_data[categorical_variables] = raw_data[categorical_variables].astype("category")

#### Target Variable

The original categorical target variable with three categories should be converted to a binary variable, as previously mentioned in the data audit.

In [19]:
# Create binary target
raw_data["readmitted30"] = (
    raw_data["readmitted"] == "<30"
).astype(int)

# Remove original target
raw_data = raw_data.drop(columns=["readmitted"])

#### Missing Values

Variables with extensive missingness should be removed, based on the data audit.

In [19]:
raw_data = raw_data.drop(columns=["weight", "payer_code"])

Other variables relating to different diagnoses should not be removed or imputed as there may be reason for such values to be missing. Further diagnoses may not always be reached.

In [ ]:
from sklearn.impute import SimpleImputer

race_imputer = SimpleImputer(
    strategy="constant",
    fill_value="Unknown"
)

diagnosis_imputer = SimpleImputer(
    strategy="constant",
    fill_value="Unknown"
)

# Identify columns
race_col = ["race"]
diagnosis_cols = ["diag_1", "diag_2", "diag_3"]

# missingness flag
for col in diagnosis_cols:
    X_train[f"{col}_missing"] = X_train[col].isna().astype(int)
    X_val[f"{col}_missing"] = X_val[col].isna().astype(int)
    X_test[f"{col}_missing"] = X_test[col].isna().astype(int)

# Fit on training data and transform
X_train[race_col] = race_imputer.fit_transform(X_train[race_col])
X_val[race_col] = race_imputer.transform(X_val[race_col])
X_test[race_col] = race_imputer.transform(X_test[race_col])

X_train[diagnosis_cols] = diagnosis_imputer.fit_transform(X_train[diagnosis_cols])
X_val[diagnosis_cols] = diagnosis_imputer.transform(X_val[diagnosis_cols])
X_test[diagnosis_cols] = diagnosis_imputer.transform(X_test[diagnosis_cols])

#### Scaling numerical features

In [ ]:
log_cols = [
    "num_lab_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

# Training data
X_train[log_cols] = np.log1p(X_train[log_cols])

# Validation data
X_val[log_cols] = np.log1p(X_val[log_cols])

# Test data
X_test[log_cols] = np.log1p(X_test[log_cols])

#### Features, Target Variable and Patients

Before separating into sets, the target variable and features should be identified.

In [21]:
# Target variable
y = raw_data["readmitted_30"]

# Patients
patients = raw_data["patient_nbr"]

# Features
X = raw_data.drop(
    columns=["readmitted_30", "patient_nbr"]
)


#### Training, Test & Validation Sets

Now, the data can be split into training, test and validation sets. A group aware split is carried out, to prevent data leakage. The split is 70% training, 15% validation and 15% test. A k-fold cross validation should be used in this case as a group aware split has been carried out.

In [22]:
from sklearn.model_selection import GroupShuffleSplit

# group-based split
splitter1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=123
)

train_idx, temp_idx = next(
    splitter1.split(X, y, groups=patients)
)

splitter2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=123
)

val_relative_idx, test_relative_idx = next(
    splitter2.split(
        X.iloc[temp_idx],
        y.iloc[temp_idx],
        groups=patients.iloc[temp_idx]
    )
)

val_idx = temp_idx[val_relative_idx]
test_idx = temp_idx[test_relative_idx]

# Training
X_train = X.iloc[train_idx].copy()
y_train = y.iloc[train_idx].copy()

# Validation
X_val = X.iloc[val_idx].copy()
y_val = y.iloc[val_idx].copy()

# Test
X_test = X.iloc[test_idx].copy()
y_test = y.iloc[test_idx].copy()

#### Categorical Variable Encoding

In [23]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

columns_to_remove = ["encounter_id", "diag_1", "diag_2", "diag_3"]

X_train = X_train.drop(columns=columns_to_remove)
X_val = X_val.drop(columns=columns_to_remove)
X_test = X_test.drop(columns=columns_to_remove)

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "str"]
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"),
         categorical_cols)
    ],
    remainder="passthrough"
)

# Fit on training data only
X_train_encoded = preprocessor.fit_transform(X_train)

# Apply to validation and test data
X_val_encoded = preprocessor.transform(X_val)
X_test_encoded = preprocessor.transform(X_test)

In [24]:
from pathlib import Path
from scipy.sparse import issparse

processed = Path(
    r"C:\Users\faith\OneDrive\Documents\MDX1_MSc_Health_Data_Science\CT4101_Machine_Learning\Machine_Learning_Portfolio\CT4101-Machine-Learning-Portfolio\data\processed"
)

# Save datasets
datasets = {
    "X_train.csv": X_train_encoded,
    "X_val.csv": X_val_encoded,
    "X_test.csv": X_test_encoded,
    "y_train.csv": y_train,
    "y_val.csv": y_val,
    "y_test.csv": y_test
}

for filename, data in datasets.items():

    if issparse(data):
        data = pd.DataFrame.sparse.from_spmatrix(data)

    elif not isinstance(data, pd.DataFrame):
        data = pd.DataFrame(data)

    data.reset_index(drop=True).to_csv(
        processed / filename,
        index=False
    )
